# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook is the full-depth leakage audit for **Lane 2 — Refresh / Content Opportunity Scoring**.

It builds the feature vector on real warehouse data (`month=2026-03`), classifies every column,
hunts **all three types** of leakage from the taxonomy (label-derived, future-window,
decision-derived), demonstrates each with a train-with vs train-without test, and compares
random vs grouped (client-holdout) splits.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `hunting-leakage-and-validating` + `flyrank/flyrank-data` for this task.

---

### Key design decision — where is the "decision moment"?

```
  Mar 1 ──────── Mar 15 ──────── Mar 31
  │   FEATURE WINDOW   │   LABEL WINDOW    │
  │   (the "past")     │   (the "future")  │
  │   knowable now     │   not known yet   │
                  ↑
          DECISION MOMENT
          (midnight March 15)
```

**A feature is legal if and only if it can be computed using ONLY rows where `report_date <= '2026-03-15'`.**
Anything touching March 16-31 is future information — even partial totals, even ratios that include it.

In [1]:
%pip -q install duckdb huggingface_hub requests scikit-learn

In [2]:
import os, getpass

# Use Colab Secrets (🔑 panel → HF_TOKEN), then env var, then safe prompt.
# NEVER paste a token into a code cell — this repo is public.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

print('Token loaded:', 'YES ✅' if HF_TOKEN else 'NO ❌')

Token loaded: YES ✅


In [3]:
import requests

headers = {'Authorization': f'Bearer {HF_TOKEN}'}
r_who = requests.get('https://huggingface.co/api/whoami-v2', headers=headers, timeout=10)
if r_who.status_code == 200:
    print(f'✅ Token valid. HF account: {r_who.json().get("name", "unknown")}')
else:
    raise RuntimeError(
        f'❌ Token rejected (HTTP {r_who.status_code}).\n'
        'Fix: create a plain READ token at https://huggingface.co/settings/tokens'
    )

r_gate = requests.get('https://huggingface.co/api/datasets/FlyRank/internship-warehouse',
                      headers=headers, timeout=10)
if r_gate.status_code == 200:
    print('✅ Gate accepted — warehouse access confirmed.')
elif r_gate.status_code == 403:
    raise RuntimeError(
        '❌ Gate not accepted. Accept at https://huggingface.co/datasets/FlyRank/internship-warehouse'
    )
else:
    print(f'⚠️  Unexpected status {r_gate.status_code}')

✅ Token valid. HF account: Artasam-Khan
✅ Gate accepted — warehouse access confirmed.


In [4]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MARCH  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Smoke-test on smallest table before touching the 79M-row fact table.
n = con.sql(f"SELECT COUNT(*) FROM {DIM_CLIENTS}").fetchone()[0]
print(f'✅ DuckDB auth confirmed. dim_clients: {n} rows.')
print('Using mid-panel month: 2026-03')

✅ DuckDB auth confirmed. dim_clients: 104 rows.
Using mid-panel month: 2026-03


---

## 1. Build the feature vector

Every feature is built **exclusively** from rows where `report_date <= '2026-03-15'`
(the feature window). The label is built from rows where `report_date > '2026-03-15'`
(the label window). These two windows never overlap.

I also pull `imp_last15` and `ga4_sessions_march` into the dataframe — **not** as features, but
as deliberate suspects for the leakage hunts in Section 3. They are never in `SAFE_FEATURES`.

**The 6 safe features:**
| Feature | Built from | Why safe |
|---|---|---|
| `prev_impressions` | Mar 1-15 sum | Past visibility — decision moment is Mar 15 |
| `prev_clicks` | Mar 1-15 sum | Past clicks — observable before Mar 15 |
| `prev_avg_position` | Mar 1-15 avg (pos > 0) | Past rank — Search Console reports it daily |
| `prev_days_active` | Mar 1-15 count | Consistency — countable at Mar 15 |
| `log_prev_impressions` | `log1p(prev_impressions)` | Compresses heavy right tail — same window |
| `prev_ctr` | `prev_clicks / prev_impressions` | CTR from the feature window only |

**Label:** `is_declining_proxy = (imp_last15 < 0.8 × prev_impressions)`  
A >20% drop in Mar 16-31 impressions relative to Mar 1-15. This is a **proxy label** —
it is not a forward-looking outcome, it is a rule applied to current data.

In [5]:
# Build aggregated feature vector from the daily fact table (month=2026-03).
# One row per page. Feature window = Mar 1-15. Label window = Mar 16-31.
# SUSPECT columns (imp_last15, ga4_sessions_march) are pulled for leakage tests only.

raw = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- ============================================================
        -- SAFE FEATURES: strictly from the feature window (Mar 1-15)
        -- ============================================================
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END)
                                                                        AS prev_impressions,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END)
                                                                        AS prev_clicks,
        AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0
            THEN gsc_avg_position END)                                  AS prev_avg_position,
        SUM(CASE WHEN report_date <= '2026-03-15' AND gsc_impressions > 0
            THEN 1 ELSE 0 END)                                          AS prev_days_active,

        -- ============================================================
        -- LABEL COMPONENT: from the label window (Mar 16-31)
        -- Never a feature. Kept here only to compute is_declining_proxy.
        -- ============================================================
        SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_impressions ELSE 0 END)
                                                                        AS imp_last15,

        -- ============================================================
        -- SUSPECT columns (for leakage hunt tests)
        -- ============================================================
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)
                                                                        AS ga4_sessions_march

    FROM {FACT_MARCH}
    GROUP BY content_hash_id, client_hash_id
    HAVING prev_impressions >= 50    -- minimum signal in the FEATURE window
""").df()

# --- Engineered features (still from feature window only) ---
raw['log_prev_impressions'] = np.log1p(raw['prev_impressions'])
raw['prev_ctr'] = raw['prev_clicks'] / (raw['prev_impressions'] + 1)   # +1 avoids zero div

# --- Missing value handling ---
# prev_avg_position is NaN for pages that had no position data at all in Mar 1-15.
# Fill with 50.0 = deep/invisible position (conservative assumption).
raw['prev_avg_position'] = raw['prev_avg_position'].fillna(50.0)

# --- Proxy label ---
raw['is_declining_proxy'] = (raw['imp_last15'] < 0.8 * raw['prev_impressions']).astype(int)

# --- Define the ONLY allowed feature list ---
# No column outside this list may enter a model training call.
SAFE_FEATURES = [
    'prev_impressions',
    'prev_clicks',
    'prev_avg_position',
    'prev_days_active',
    'log_prev_impressions',
    'prev_ctr'
]

print(f"Feature vector: {len(raw):,} pages × {raw.shape[1]} columns")
print(f"Base rate: {raw['is_declining_proxy'].mean():.1%} declining (majority class = {max(raw['is_declining_proxy'].mean(), 1 - raw['is_declining_proxy'].mean()):.1%})")
print(f"Clients in dataset: {raw['client_hash_id'].nunique()}")
print(f"\nSafe features ({len(SAFE_FEATURES)}): {SAFE_FEATURES}")
print()
print(raw[SAFE_FEATURES + ['is_declining_proxy']].describe().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector: 92,548 pages × 11 columns
Base rate: 28.6% declining (majority class = 71.4%)
Clients in dataset: 40

Safe features (6): ['prev_impressions', 'prev_clicks', 'prev_avg_position', 'prev_days_active', 'log_prev_impressions', 'prev_ctr']

       prev_impressions  prev_clicks  prev_avg_position  prev_days_active  \
count         92548.000    92548.000          92548.000         92548.000   
mean           1368.890        4.124             14.022            14.049   
std            3323.195       16.774             14.313             2.216   
min              50.000        0.000              0.041             1.000   
25%             143.000        0.000              4.675            14.000   
50%             406.000        1.000              8.244            15.000   
75%            1232.000        3.000             18.458            15.000   
max          161575.000     2395.000            127.621            15.000   

       log_prev_impressions   prev_ctr  is_declining_pr

---

## 2. Feature notes — meaning, missing, available-when?

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `prev_impressions` | Total GSC impressions Mar 1-15 | No missing: `HAVING >= 50` guarantees a value | ✅ Knowable at Mar 15 — pure past count |
| `prev_clicks` | Total GSC clicks Mar 1-15 | No missing: zero clicks is a valid count | ✅ Knowable at Mar 15 |
| `prev_avg_position` | Mean GSC rank over days with position data (Mar 1-15) | Filled with `50.0` (deep position) when no position data at all | ✅ Knowable at Mar 15 |
| `prev_days_active` | Count of days (Mar 1-15) with ≥ 1 impression | No missing: integer count 0-15 | ✅ Knowable at Mar 15 — consistency signal |
| `log_prev_impressions` | `log1p(prev_impressions)` | No missing: derived from prev_impressions | ✅ Knowable at Mar 15 — same window |
| `prev_ctr` | `prev_clicks / (prev_impressions + 1)` | No missing: +1 prevents div-by-zero | ✅ Knowable at Mar 15 — same window |

### Timeline diagram

```
  Mar 1 ──────────── Mar 15 ─────────────── Mar 31
  │                      │                       │
  │   FEATURE WINDOW     │     LABEL WINDOW      │
  │   prev_impressions   │     imp_last15         │
  │   prev_clicks        │     (NOT a feature)    │
  │   prev_avg_position  │                        │
  │   prev_days_active   │  Label:                │
  │   log_prev_impr      │  is_declining_proxy =  │
  │   prev_ctr           │  imp_last15 < 0.8 *    │
  │                      │  prev_impressions      │
  │                      ↑                        │
  │               DECISION MOMENT                  │
  │             (midnight Mar 15)                  │
```

Every feature is strictly knowable **before** the prediction moment. The label component
(`imp_last15`) is in the dataframe for transparency and for leakage tests, but is **never**
in `SAFE_FEATURES`.

In [6]:
# Verify: no NaNs in safe features after our handling.
missing = raw[SAFE_FEATURES].isna().sum()
print("Missing values per safe feature (should all be 0):")
print(missing.to_string())
print(f"\nTotal NaN cells in safe features: {missing.sum()}")

Missing values per safe feature (should all be 0):
prev_impressions        0
prev_clicks             0
prev_avg_position       0
prev_days_active        0
log_prev_impressions    0
prev_ctr                0

Total NaN cells in safe features: 0


---

## 3. The leakage hunt

The `hunting-leakage-and-validating/SKILL.md` defines three leakage types. I test all three.

**Methodology** (from the skill): *"Deliberately ADD a leaky feature and watch the score jump
toward 1.0 — if it doesn't, your test harness itself is broken. Then remove it and keep the
honest number."*

---

### Hunt 1: Label-derived features (Leakage Type 1)

**Suspect:** `imp_last15` — the numerator of the label formula.  
**Why suspicious:** `is_declining_proxy = (imp_last15 < 0.8 × prev_impressions)`.  
The label IS `imp_last15`. Giving the model `imp_last15` is giving it the answer.  
**Test:** train WITH it vs WITHOUT. The skill says: *"collapse from ~1.0 to ~0.7 is the confession."*

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

df = raw.dropna(subset=SAFE_FEATURES).copy()
y  = df['is_declining_proxy']

# -------------------------------------------------------
# Test A: SAFE features only (honest baseline)
# -------------------------------------------------------
X_safe = df[SAFE_FEATURES]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_safe, y, test_size=0.25, random_state=42, stratify=y
)
rf_safe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_safe.fit(X_tr, y_tr)
pred_safe = rf_safe.predict(X_te)

prec_safe = precision_score(y_te, pred_safe)
rec_safe  = recall_score(y_te, pred_safe)
f1_safe   = f1_score(y_te, pred_safe)

# -------------------------------------------------------
# Test B: SAFE + imp_last15 (label numerator — LEAKED)
# -------------------------------------------------------
LEAKED_1 = SAFE_FEATURES + ['imp_last15']
X_leaked_1 = df[LEAKED_1]
X_tr_l1, X_te_l1, y_tr_l1, y_te_l1 = train_test_split(
    X_leaked_1, y, test_size=0.25, random_state=42, stratify=y
)
rf_leaked_1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaked_1.fit(X_tr_l1, y_tr_l1)
pred_l1 = rf_leaked_1.predict(X_te_l1)

prec_l1 = precision_score(y_te_l1, pred_l1)
rec_l1  = recall_score(y_te_l1, pred_l1)
f1_l1   = f1_score(y_te_l1, pred_l1)

print("=" * 68)
print("HUNT 1: Label-derived feature — imp_last15 (label numerator)")
print("=" * 68)
print(f"{'Metric':<20} {'Safe (honest)':<22} {'+ imp_last15 (LEAKED)'}")
print(f"{'-'*20} {'-'*22} {'-'*22}")
print(f"{'Precision':<20} {prec_safe:<22.3f} {prec_l1:.3f}")
print(f"{'Recall':<20} {rec_safe:<22.3f} {rec_l1:.3f}")
print(f"{'F1':<20} {f1_safe:<22.3f} {f1_l1:.3f}")
print(f"{'Base rate':<20} {y_te.mean():<22.3f} {y_te_l1.mean():.3f}")
print()
print("VERDICT: imp_last15 is directly inside the label formula.")
print("         is_declining_proxy = (imp_last15 < 0.8 × prev_impressions)")
print("         Score jump to ~1.0 is the confession — the model has the answer.")
print("         In the real world, imp_last15 does not exist at decision time (Mar 15).")
print("ACTION:  imp_last15 EXCLUDED from all modeling.")

del rf_leaked_1  # destroy the leaked model — only rf_safe is valid

HUNT 1: Label-derived feature — imp_last15 (label numerator)
Metric               Safe (honest)          + imp_last15 (LEAKED)
-------------------- ---------------------- ----------------------
Precision            0.387                  0.986
Recall               0.269                  0.968
F1                   0.317                  0.977
Base rate            0.286                  0.286

VERDICT: imp_last15 is directly inside the label formula.
         is_declining_proxy = (imp_last15 < 0.8 × prev_impressions)
         Score jump to ~1.0 is the confession — the model has the answer.
         In the real world, imp_last15 does not exist at decision time (Mar 15).
ACTION:  imp_last15 EXCLUDED from all modeling.


### Hunt 2: Future/overlapping window features (Leakage Type 2)

**Suspect:** A hypothetical `total_march_impressions` = SUM over all of March (Mar 1-31).  
**Why it's a suspect:** The skill says: *"A feature summed over a window that CONTAINS
the label's window already knows the outcome."*

If someone used `SUM(gsc_impressions)` over the full month (Mar 1-31) as a feature — as
I initially did in the w03 contract notebook — they would be including Mar 16-31 data inside
a feature. The label is also defined on Mar 16-31. The feature window overlaps the label window.

**Test:** Build a full-month sum and compare it against the feature-window-only `prev_impressions`.

In [8]:
# Build the overlapping-window suspect: total_march_impressions (Mar 1-31)
# This includes the label window (Mar 16-31) — it SEES the future.

total_march = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_march_impressions   -- ← full month: contains the label window!
    FROM {FACT_MARCH}
    GROUP BY content_hash_id, client_hash_id
""").df()

# Merge into our feature frame
df_hunt2 = df.merge(total_march, on=['content_hash_id', 'client_hash_id'], how='inner')

LEAKED_2 = SAFE_FEATURES + ['total_march_impressions']
X_leaked_2 = df_hunt2[LEAKED_2]
y_hunt2    = df_hunt2['is_declining_proxy']

X_tr_l2, X_te_l2, y_tr_l2, y_te_l2 = train_test_split(
    X_leaked_2, y_hunt2, test_size=0.25, random_state=42, stratify=y_hunt2
)
rf_leaked_2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaked_2.fit(X_tr_l2, y_tr_l2)
pred_l2 = rf_leaked_2.predict(X_te_l2)

prec_l2 = precision_score(y_te_l2, pred_l2)
rec_l2  = recall_score(y_te_l2, pred_l2)
f1_l2   = f1_score(y_te_l2, pred_l2)

# Also test honest model on the same filtered subset for fair comparison
X_safe_h2 = df_hunt2[SAFE_FEATURES]
X_tr_h2, X_te_h2, y_tr_h2, y_te_h2 = train_test_split(
    X_safe_h2, y_hunt2, test_size=0.25, random_state=42, stratify=y_hunt2
)
rf_safe_h2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_safe_h2.fit(X_tr_h2, y_tr_h2)
pred_h2    = rf_safe_h2.predict(X_te_h2)
prec_h2 = precision_score(y_te_h2, pred_h2)
rec_h2  = recall_score(y_te_h2, pred_h2)
f1_h2   = f1_score(y_te_h2, pred_h2)

print("=" * 68)
print("HUNT 2: Overlapping-window feature — total_march_impressions")
print("=" * 68)
print(f"{'Metric':<20} {'Safe (honest)':<22} {'+ full-month total (LEAKED)'}")
print(f"{'-'*20} {'-'*22} {'-'*26}")
print(f"{'Precision':<20} {prec_h2:<22.3f} {prec_l2:.3f}")
print(f"{'Recall':<20} {rec_h2:<22.3f} {rec_l2:.3f}")
print(f"{'F1':<20} {f1_h2:<22.3f} {f1_l2:.3f}")
print(f"{'Base rate':<20} {y_te_h2.mean():<22.3f} {y_te_l2.mean():.3f}")
print()
print("VERDICT: total_march_impressions = Mar 1-15 + Mar 16-31.")
print("         The Mar 16-31 portion IS the label window — the feature sees the future.")
print("         This is how the w03 contract notebook originally had leakage: using")
print("         'total_impressions' (full month) instead of 'prev_impressions' (Mar 1-15).")
print("ACTION:  Any full-month aggregate is EXCLUDED. Only prev_* features are legal.")

del rf_leaked_2, rf_safe_h2

HUNT 2: Overlapping-window feature — total_march_impressions
Metric               Safe (honest)          + full-month total (LEAKED)
-------------------- ---------------------- --------------------------
Precision            0.387                  0.978
Recall               0.269                  0.932
F1                   0.317                  0.954
Base rate            0.286                  0.286

VERDICT: total_march_impressions = Mar 1-15 + Mar 16-31.
         The Mar 16-31 portion IS the label window — the feature sees the future.
         This is how the w03 contract notebook originally had leakage: using
         'total_impressions' (full month) instead of 'prev_impressions' (Mar 1-15).
ACTION:  Any full-month aggregate is EXCLUDED. Only prev_* features are legal.


### Hunt 3: Decision-derived features / product flags (Leakage Type 3)

**Suspect:** `ga4_sessions_march` — GA4 session counts from March.  
**Why suspicious:** The flyrank-data skill warns: *"Rows before a client's `ga4_data_start`
have GA4 columns zero-FILLED with `ga4_data_available = FALSE` — those zeros mean
'not measured', not 'no engagement'.*"  

Using GA4 sessions as a feature means the model learns **which clients have Analytics tracking**,
not which pages are actually declining in search. The tracking start date is a product-team decision,
not a search-performance signal — that makes it decision-derived.

The skill says: *"Decision-derived features encode a decision someone already made. Using them
as features means learning the old rule, not the world — a circular result."*

In [9]:
# -------------------------------------------------------
# Test D: SAFE + ga4_sessions_march (decision-derived suspect)
# -------------------------------------------------------
LEAKED_3   = SAFE_FEATURES + ['ga4_sessions_march']
X_leaked_3 = df[LEAKED_3]

X_tr_l3, X_te_l3, y_tr_l3, y_te_l3 = train_test_split(
    X_leaked_3, y, test_size=0.25, random_state=42, stratify=y
)
rf_leaked_3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaked_3.fit(X_tr_l3, y_tr_l3)
pred_l3 = rf_leaked_3.predict(X_te_l3)

prec_l3 = precision_score(y_te_l3, pred_l3)
rec_l3  = recall_score(y_te_l3, pred_l3)
f1_l3   = f1_score(y_te_l3, pred_l3)

# Show how much of the data even has GA4 — to prove the sparsity point
pct_ga4 = (df['ga4_sessions_march'] > 0).mean()

print("=" * 68)
print("HUNT 3: Decision-derived feature — ga4_sessions_march")
print("=" * 68)
print(f"% of pages with any GA4 sessions in March: {pct_ga4:.1%}  (the rest are zero-filled)")
print()
print(f"{'Metric':<20} {'Safe (honest)':<22} {'+ ga4_sessions (SUSPECT)'}")
print(f"{'-'*20} {'-'*22} {'-'*24}")
print(f"{'Precision':<20} {prec_safe:<22.3f} {prec_l3:.3f}")
print(f"{'Recall':<20} {rec_safe:<22.3f} {rec_l3:.3f}")
print(f"{'F1':<20} {f1_safe:<22.3f} {f1_l3:.3f}")
print(f"{'Base rate':<20} {y_te.mean():<22.3f} {y_te_l3.mean():.3f}")
print()
print("VERDICT: GA4 sessions in March are not a search performance signal.")
print("         They are near-zero for most clients because GA4 tracking has not started.")
print("         The model would learn 'has tracking = client profile', not 'page is declining'.")
print("         This is a decision-derived feature: the tracking schedule is a product decision.")
print("ACTION:  ga4_sessions_march EXCLUDED. Filter on ga4_data_available IS TRUE before use.")

del rf_leaked_3

HUNT 3: Decision-derived feature — ga4_sessions_march
% of pages with any GA4 sessions in March: 55.1%  (the rest are zero-filled)

Metric               Safe (honest)          + ga4_sessions (SUSPECT)
-------------------- ---------------------- ------------------------
Precision            0.387                  0.428
Recall               0.269                  0.269
F1                   0.317                  0.330
Base rate            0.286                  0.286

VERDICT: GA4 sessions in March are not a search performance signal.
         They are near-zero for most clients because GA4 tracking has not started.
         The model would learn 'has tracking = client profile', not 'page is declining'.
         This is a decision-derived feature: the tracking schedule is a product decision.
ACTION:  ga4_sessions_march EXCLUDED. Filter on ga4_data_available IS TRUE before use.


---

### Hunt 4: Random split vs client-holdout split

A random train/test split lets pages from the same client appear in **both** train and test sets.
If Client A has unusual traffic patterns, the model can memorize "Client A's pages tend to decline"
and get credit for it — even though it learned nothing generalizable.

The skill says: *"The honest question is 'does it work on a group it never saw?'"*  
The gap between random-split score and grouped-split score **is itself a finding** about
how much client memorization was happening.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

# -------------------------------------------------------
# Client-holdout split: train on some clients, test on DIFFERENT clients
# -------------------------------------------------------
X_all  = df[SAFE_FEATURES]
y_all  = df['is_declining_proxy']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_all, y_all, groups))

X_tr_g, X_te_g = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_tr_g, y_te_g = y_all.iloc[train_idx], y_all.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_grouped.fit(X_tr_g, y_tr_g)
pred_grouped = rf_grouped.predict(X_te_g)

prec_grouped = precision_score(y_te_g, pred_grouped)
rec_grouped  = recall_score(y_te_g, pred_grouped)
f1_grouped   = f1_score(y_te_g, pred_grouped)

n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients  = groups.iloc[test_idx].nunique()

print("=" * 68)
print("HUNT 4: Random split vs client-holdout split")
print("=" * 68)
print(f"Random split   — train pages: {len(X_tr):,}  | test pages: {len(X_te):,}")
print(f"Grouped split  — train clients: {n_train_clients}  | test clients: {n_test_clients}")
print(f"                 train pages: {len(train_idx):,}  | test pages: {len(test_idx):,}")
print()
print(f"{'Metric':<20} {'Random split':<22} {'Client-holdout split'}")
print(f"{'-'*20} {'-'*22} {'-'*22}")
print(f"{'Precision':<20} {prec_safe:<22.3f} {prec_grouped:.3f}")
print(f"{'Recall':<20} {rec_safe:<22.3f} {rec_grouped:.3f}")
print(f"{'F1':<20} {f1_safe:<22.3f} {f1_grouped:.3f}")
print(f"{'Base rate':<20} {y_te.mean():<22.3f} {y_te_g.mean():.3f}")
print()
gap = f1_safe - f1_grouped
print(f"Gap (random F1 − grouped F1): {gap:+.3f}")
if gap > 0.05:
    print("⚠️  Gap > 0.05: the random split flatters the model. Some 'skill' is client memorization.")
    print("    Use the grouped-split number as the honest result going forward.")
elif gap > 0.02:
    print("⚠️  Small gap (0.02-0.05): mild client memorization. Not catastrophic but noted.")
else:
    print("✅ Minimal gap (< 0.02): the model generalizes reasonably across unseen clients.")
print()
print("Going forward: client-holdout is the honest split for ALL evaluation.")

HUNT 4: Random split vs client-holdout split
Random split   — train pages: 69,411  | test pages: 23,137
Grouped split  — train clients: 30  | test clients: 10
                 train pages: 67,662  | test pages: 24,886

Metric               Random split           Client-holdout split
-------------------- ---------------------- ----------------------
Precision            0.387                  0.450
Recall               0.269                  0.209
F1                   0.317                  0.285
Base rate            0.286                  0.369

Gap (random F1 − grouped F1): +0.032
⚠️  Small gap (0.02-0.05): mild client memorization. Not catastrophic but noted.

Going forward: client-holdout is the honest split for ALL evaluation.


### Feature importance sanity check

The skill's attack checklist says: *"Top feature importance sanity-checked — 'too good'
investigated, not celebrated."* If any single feature towers over all others (>50% importance),
we need to ask: is it structurally entangled with the label?

In [11]:
# Feature importances from the HONEST grouped-split model (rf_grouped)
importances = pd.Series(rf_grouped.feature_importances_, index=SAFE_FEATURES)
importances = importances.sort_values(ascending=False)

print("Feature importances (client-holdout model, descending):")
print()
for feat, imp in importances.items():
    bar = '█' * int(imp * 50)
    print(f"  {feat:<25} {imp:.3f}  {bar}")

top_feat = importances.index[0]
top_imp  = importances.iloc[0]
print()
if top_imp > 0.5:
    print(f"⚠️  {top_feat} dominates ({top_imp:.1%}). Investigate:")
    print("   Is this feature structurally related to the label formula?")
    print("   Check: does removing it collapse the score? If yes, it may be a subtle leak.")
else:
    print(f"✅ No single feature dominates. Top feature ({top_feat}: {top_imp:.1%})")
    print("   carries signal but doesn't tower over the rest. This is a healthy distribution.")

Feature importances (client-holdout model, descending):

  prev_avg_position         0.462  ███████████████████████
  prev_impressions          0.182  █████████
  log_prev_impressions      0.181  █████████
  prev_ctr                  0.110  █████
  prev_days_active          0.038  █
  prev_clicks               0.028  █

✅ No single feature dominates. Top feature (prev_avg_position: 46.2%)
   carries signal but doesn't tower over the rest. This is a healthy distribution.


---

## 4. What I excluded and why

| Column | Leakage type | Why excluded |
|---|---|---|
| `imp_last15` | **Label-derived (Type 1)** | Directly in the label formula: `is_declining_proxy = (imp_last15 < 0.8 × prev_impressions)`. Hunt 1 proved it: score jumped toward 1.0 |
| `total_march_impressions` (full month) | **Future-window (Type 2)** | Sums Mar 1-31, which contains the label window (Mar 16-31). Hunt 2 showed the score boost. Only `prev_*` versions are legal |
| `ga4_sessions_march` | **Decision-derived (Type 3)** | GA4 tracking is a product-team decision; rows without tracking are zero-filled. Using it means learning the tracking schedule, not page performance |
| All GA4 columns when `ga4_data_available IS NOT TRUE` | **Decision-derived (Type 3)** | Zeros mean "not measured yet", not "no engagement". Systematically misleading |
| `ga4_data_available` flag itself | **Decision-derived (Type 3)** | Encodes which clients have Analytics, not which pages are declining. Never a feature |
| `content_hash_id` | **Context** | Pseudonymous ID — for grouping/joining only |
| `client_hash_id` | **Context** | Used for grouped splits only. Including it as a feature means memorizing clients |
| `momentum_ratio = imp_last15 / (imp_prev15 + 1)` | **Label-derived (Type 1)** | Uses `imp_last15` (from the label window) in its numerator. Excluded from SAFE_FEATURES |
| FlyRank product flags (`health_score`, `needs_ctr_fix`, etc.) | **Decision-derived (Type 3)** | Not in the dataset by design. If present, they would encode the existing product rule — a circular result |

In [12]:
# The attack checklist from the skill file — every item verified.
base_rate = y_all.mean()
naive_baseline = max(base_rate, 1 - base_rate)

print("THE ATTACK CHECKLIST — hunting-leakage-and-validating/SKILL.md")
print("=" * 65)
print("[✅] Timeline drawn: features from Mar 1-15 only; label from Mar 16-31")
print("[✅] No label-derived columns in features (imp_last15 tested & removed) — Hunt 1")
print("[✅] No overlapping-window features (full-month totals tested & removed) — Hunt 2")
print("[✅] No product flags / decision-derived features (GA4 tested & removed) — Hunt 3")
print("[✅] Split grouped by client_hash_id (GroupShuffleSplit) — Hunt 4")
print(f"[✅] Base rate printed: {base_rate:.1%} declining")
print(f"     Naive majority-class baseline: {naive_baseline:.1%}")
print("[✅] Top feature importance sanity-checked")
print("[✅] Metrics recomputed out-of-fold on test set only (never in-sample)")
print()
print(f"Final safe feature set ({len(SAFE_FEATURES)}): {SAFE_FEATURES}")
print(f"Final honest evaluation: client-holdout split")
print(f"  Precision : {prec_grouped:.3f}")
print(f"  Recall    : {rec_grouped:.3f}")
print(f"  F1        : {f1_grouped:.3f}")
print(f"  Base rate : {y_te_g.mean():.3f}  |  Naive baseline: {max(y_te_g.mean(), 1-y_te_g.mean()):.3f}")

THE ATTACK CHECKLIST — hunting-leakage-and-validating/SKILL.md
[✅] Timeline drawn: features from Mar 1-15 only; label from Mar 16-31
[✅] No label-derived columns in features (imp_last15 tested & removed) — Hunt 1
[✅] No overlapping-window features (full-month totals tested & removed) — Hunt 2
[✅] No product flags / decision-derived features (GA4 tested & removed) — Hunt 3
[✅] Split grouped by client_hash_id (GroupShuffleSplit) — Hunt 4
[✅] Base rate printed: 28.6% declining
     Naive majority-class baseline: 71.4%
[✅] Top feature importance sanity-checked
[✅] Metrics recomputed out-of-fold on test set only (never in-sample)

Final safe feature set (6): ['prev_impressions', 'prev_clicks', 'prev_avg_position', 'prev_days_active', 'log_prev_impressions', 'prev_ctr']
Final honest evaluation: client-holdout split
  Precision : 0.450
  Recall    : 0.209
  F1        : 0.285
  Base rate : 0.369  |  Naive baseline: 0.631


---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (all IDs are pseudonymized hashes)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] All three leakage types from the taxonomy tested: label-derived, future-window, decision-derived
- [x] Random split vs grouped split compared; gap reported and interpreted
- [x] Feature importances sanity-checked from the grouped-split model
- [x] Full attack checklist completed with base rate printed next to every metric
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.